# Glicko-2 leaderboard (series-based)

This notebook displays a compact leaderboard from `data/metrics/glicko2_latest_all.parquet`.

Notes:
- Ratings are computed on **series outcomes** (keyed by `(leagueid, series_id)`), not per-map.
- If you changed `glicko2_config.json`, rerun `make precompute GLICKO_CONFIG=glicko2_config.json` before opening this notebook.


In [1]:
from pathlib import Path
import sys

import polars as pl
import pandas as pd


def find_root() -> Path:
    cand = Path.cwd()
    for c in [cand, *cand.parents]:
        if (c / "src").exists() and (c / "data").exists():
            return c
    return cand


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROCESSED_DIR = ROOT / "data" / "processed"
METRICS_DIR = ROOT / "data" / "metrics"

G2_LATEST_ALL_PATH = METRICS_DIR / "glicko2_latest_all.parquet"
SERIES_RESULTS_PATH = METRICS_DIR / "series_results.parquet"

ALIASES_PATH = ROOT / "data" / "team_aliases.csv"

print("ROOT:", ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("METRICS_DIR:", METRICS_DIR)
print("glicko2_latest_all.parquet exists?", G2_LATEST_ALL_PATH.exists())
print("series_results.parquet exists?", SERIES_RESULTS_PATH.exists())


ROOT: /home/ju/Documents/Dev/MachineLearning/Dota-Datas
PROCESSED_DIR: /home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/processed
METRICS_DIR: /home/ju/Documents/Dev/MachineLearning/Dota-Datas/data/metrics
glicko2_latest_all.parquet exists? True
series_results.parquet exists? True


In [2]:
def must_read_parquet(path: Path) -> pl.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run: make precompute OUT=data/processed METRICS_OUT=data/metrics (and set GLICKO_CONFIG if needed)."
        )
    return pl.read_parquet(path)


g2_latest_all = must_read_parquet(G2_LATEST_ALL_PATH)
series_results = must_read_parquet(SERIES_RESULTS_PATH)

g2_latest_all.head(3), series_results.head(3)


(shape: (3, 7)
 ┌─────────┬─────────────┬───────────┬──────────┬─────────────┬──────┬────────────┐
 │ team_id ┆ rating      ┆ rd        ┆ sigma    ┆ score       ┆ rank ┆ score_rank │
 │ ---     ┆ ---         ┆ ---       ┆ ---      ┆ ---         ┆ ---  ┆ ---        │
 │ i64     ┆ f64         ┆ f64       ┆ f64      ┆ f64         ┆ u32  ┆ u32        │
 ╞═════════╪═════════════╪═══════════╪══════════╪═════════════╪══════╪════════════╡
 │ 7119388 ┆ 2164.97338  ┆ 90.807029 ┆ 0.060036 ┆ 1983.359322 ┆ 1    ┆ 1          │
 │ 9247354 ┆ 2144.449284 ┆ 90.860557 ┆ 0.059998 ┆ 1962.728169 ┆ 2    ┆ 2          │
 │ 9823272 ┆ 2058.482785 ┆ 89.540571 ┆ 0.06007  ┆ 1879.401642 ┆ 3    ┆ 3          │
 └─────────┴─────────────┴───────────┴──────────┴─────────────┴──────┴────────────┘,
 shape: (3, 13)
 ┌──────────┬───────────┬────────────┬──────────────┬───┬──────┬─────────┬─────────────┬────────┐
 │ leagueid ┆ series_id ┆ start_time ┆ start_dt     ┆ … ┆ maps ┆ bo_type ┆ series_type ┆ weight │
 │ ---      ┆ --

In [3]:
from src.dota_data import read_processed_tables, build_team_dictionary

tables = read_processed_tables(PROCESSED_DIR)
teams_dict = build_team_dictionary(tables["matches"])

teams = (
    teams_dict.group_by("team_id")
    .agg(
        pl.col("name").drop_nulls().first().alias("team_name"),
        pl.col("tag").drop_nulls().first().alias("tag"),
        pl.col("logo_url").drop_nulls().first().alias("logo_url"),
    )
    .filter(pl.col("team_id").is_not_null())
)

teams.head(5)


team_id,team_name,tag,logo_url
i64,str,str,str
9651867,"""saadboys""","""saD""","""https://cdn.steamusercontent.c…"
9917833,"""Atlantis 2.0""","""""","""https://cdn.steamusercontent.c…"
9382454,"""COPPYBEBRA""","""COPPY""","""https://cdn.steamusercontent.c…"
8815475,"""Ghost Trick""","""GTR""","""https://cdn.steamusercontent.c…"
9337731,"""LEVIATAN""","""LEV""","""https://cdn.steamusercontent.c…"


In [4]:
# Per-team series volume and last seen date (helps interpret RD / uncertainty)
a = series_results.select(
    pl.col("team_a_id").cast(pl.Int64, strict=False).alias("team_id"),
    pl.col("start_time").cast(pl.Int64, strict=False).alias("start_time"),
).with_columns(pl.lit(1).alias("series"))
b = series_results.select(
    pl.col("team_b_id").cast(pl.Int64, strict=False).alias("team_id"),
    pl.col("start_time").cast(pl.Int64, strict=False).alias("start_time"),
).with_columns(pl.lit(1).alias("series"))

series_stats = (
    pl.concat([a, b], how="vertical")
    .drop_nulls(["team_id"])
    .group_by("team_id")
    .agg(
        pl.sum("series").alias("series_played"),
        pl.max("start_time").alias("last_start_time"),
    )
)

series_stats.head(5)


team_id,series_played,last_start_time
i64,i32,i64
9162769,1,1705312676
10026852,1,1768230676
9723788,3,1746870789
9895247,123,1769002341
9571336,1,1731507254


In [5]:
# Leaderboards
df = (
    g2_latest_all
    .join(teams, on="team_id", how="left")
    .join(series_stats, on="team_id", how="left")
)

df_pd = df.to_pandas()
if "last_start_time" in df_pd.columns:
    df_pd["last_series_dt"] = pd.to_datetime(df_pd["last_start_time"], unit="s", utc=True)

cols = [
    "score_rank",
    "rank",
    "team_id",
    "team_name",
    "rating",
    "rd",
    "score",
    "series_played",
    "last_series_dt",
]
cols = [c for c in cols if c in df_pd.columns]

TOP_N = 30

display(df_pd.sort_values("score_rank", ascending=True)[cols].head(TOP_N))

print("\nTop by raw rating (rank):")
display(df_pd.sort_values("rank", ascending=True)[cols].head(TOP_N))


,score_rank,rank,team_id,team_name,rating,rd,score,series_played,last_series_dt
0,1,1,7119388,Team Spirit,2164.973380,90.807029,1983.359322,276,2025-12-21 18:17:01+00:00
1,2,2,9247354,Team Falcons,2144.449284,90.860557,1962.728169,296,2026-01-18 20:31:21+00:00
2,3,3,9823272,Team Yandex,2058.482785,89.540571,1879.401642,72,2025-12-21 18:17:01+00:00
3,4,4,8255888,BetBoom Team,2043.587245,86.997358,1869.592530,272,2026-01-17 19:03:05+00:00
6,5,7,9338413,MOUZ,2033.207658,83.069567,1867.068524,295,2026-01-20 13:21:30+00:00
4,6,5,9572001,PVISION,2040.654800,88.538748,1863.577304,165,2026-01-05 17:22:46+00:00
5,7,6,8291895,Tundra Esports,2038.127162,90.683779,1856.759604,299,2025-12-19 19:00:00+00:00
9,8,10,8261500,Xtreme Gaming,1981.252747,89.859706,1801.533336,255,2026-01-05 14:59:36+00:00
10,9,11,36,Natus Vincere,1952.421274,78.948927,1794.523421,177,2026-01-20 17:47:13+00:00
11,10,12,2586976,OG,1934.660011,91.042383,1752.575245,244,2025-12-20 13:15:00+00:00



Top by raw rating (rank):


,score_rank,rank,team_id,team_name,rating,rd,score,series_played,last_series_dt
0,1,1,7119388,Team Spirit,2164.973380,90.807029,1983.359322,276,2025-12-21 18:17:01+00:00
1,2,2,9247354,Team Falcons,2144.449284,90.860557,1962.728169,296,2026-01-18 20:31:21+00:00
2,3,3,9823272,Team Yandex,2058.482785,89.540571,1879.401642,72,2025-12-21 18:17:01+00:00
3,4,4,8255888,BetBoom Team,2043.587245,86.997358,1869.592530,272,2026-01-17 19:03:05+00:00
4,6,5,9572001,PVISION,2040.654800,88.538748,1863.577304,165,2026-01-05 17:22:46+00:00
5,7,6,8291895,Tundra Esports,2038.127162,90.683779,1856.759604,299,2025-12-19 19:00:00+00:00
6,5,7,9338413,MOUZ,2033.207658,83.069567,1867.068524,295,2026-01-20 13:21:30+00:00
7,53,8,9547255,Bandanoone,1985.389265,282.284146,1420.820973,4,2024-09-29 18:32:32+00:00
8,15,9,9766941,FLIPSTER TALON,1984.650458,152.346431,1679.957596,23,2025-10-01 11:55:29+00:00
9,8,10,8261500,Xtreme Gaming,1981.252747,89.859706,1801.533336,255,2026-01-05 14:59:36+00:00
